In [32]:
import os, glob
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut
import xgboost as xgb
from sklearn.metrics import recall_score, precision_score, balanced_accuracy_score, roc_auc_score

In [33]:
DATA_PATH   = "/home/justine/code/Maelle05/DyslexIA/Dataset"
LABELS_PATH = "/home/justine/code/Maelle05/DyslexIA/Dataset/dyslexia_class_label.csv"

GAZE_COLS  = ['x_left', 'y_left', 'x_right', 'y_right']

files_raw = glob.glob(os.path.join(DATA_PATH, "*clean.csv"))
labels = pd.read_csv(LABELS_PATH)

In [34]:
def to_uniform(signal, timestamps):
    t_norm = (timestamps - timestamps[0]) / (timestamps[-1] - timestamps[0])
    x_new  = np.linspace(0, 1, len(signal))
    return np.interp(x_new, t_norm, signal)

def extract_features(df, gaze_cols):
    timestamps = df['t_s'].values.astype(float)
    signals = {}
    for col in gaze_cols:
        signals[col] = to_uniform(df[col].values.astype(float), timestamps)

    # Divergence binoculaire moyenne
    mean_cross_divergence_x = np.mean(np.abs(signals['x_left'] - signals['x_right']))

    # Cyclope
    x = (signals['x_left'] + signals['x_right']) / 2
    y = (signals['y_left'] + signals['y_right']) / 2

    # Vélocités
    vel_x = np.abs(np.diff(x))
    x_vel_p90p50 = np.percentile(vel_x, 90) / (np.percentile(vel_x, 50) + 1e-8)

    vel_y = np.abs(np.diff(y))
    y_vel_p90p50 = np.percentile(vel_y, 90) / (np.percentile(vel_y, 50) + 1e-8)

    # Taux de changements de direction
    dx = np.diff(x)
    x_direction_changes = np.sum(np.diff(np.sign(dx)) != 0) / len(dx)

    return np.array([mean_cross_divergence_x,
                     x_vel_p90p50, y_vel_p90p50,
                     x_direction_changes])

In [35]:
rows = []
for f in files_raw:
    df = pd.read_csv(f)
    sid = os.path.basename(f).split('_')[1]
    label_row = labels[labels['subject_id'] == sid]
    rows.append({'sid': sid,
                 'label': label_row['class_id'].values[0],
                 'features': extract_features(df, GAZE_COLS)})

sids = np.array([r['sid']      for r in rows])
X    = np.array([r['features'] for r in rows])
y    = np.array([r['label']    for r in rows])
print(f"X : {X.shape} | Classes : {np.bincount(y)} (0=non-dys, 1=dys)")

X : (255, 4) | Classes : [123 132] (0=non-dys, 1=dys)


In [ ]:
import random

RNG = np.random.RandomState(42)
N_ITER = 50  # nombre de combinaisons random testées dans l'inner LOO

def sample_params(rng):
    return {
        'max_depth': rng.randint(2, 4),
        'n_estimators': rng.randint(100, 301),
        'learning_rate': rng.uniform(0.01, 0.1),
        'reg_lambda': rng.uniform(1.0, 3.0)
    }

# Tire les N_ITER combinaisons une seule fois — identiques pour chaque fold outer
# (reproductibilité + pas de biais par re-tirage)
PARAM_LIST = [sample_params(RNG) for _ in range(N_ITER)]


loo_outer = LeaveOneOut()
y_true, y_pred, y_scores = [], [], []
thresholds_used = []
best_params_per_fold = []

for fold_i, (train_idx, test_idx) in enumerate(loo_outer.split(X, y)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_test,  y_test  = X[test_idx],  y[test_idx]

    loo_inner = LeaveOneOut()

    # -- Pour chaque combinaison de params, accumule les scores inner LOO --
    param_scores  = np.zeros(N_ITER)   # bal_acc agrégée sur les 254 inner folds
    inner_true_all = None              # partagé entre toutes les combinaisons

    # On collecte les proba pour chaque (inner_fold, param_combo) en une passe
    # pour ne faire qu'un seul inner LOO et éviter 40 passes
    n_inner = len(X_train)
    all_inner_scores = np.zeros((N_ITER, n_inner))
    inner_true_all   = np.zeros(n_inner, dtype=int)

    for j, (inner_train_idx, inner_val_idx) in enumerate(loo_inner.split(X_train, y_train)):
        inner_true_all[j] = y_train[inner_val_idx][0]
        for p_i, params in enumerate(PARAM_LIST):
            clf_inner = xgb.XGBClassifier(
                **params,
                random_state=42, verbosity=0
            )
            clf_inner.fit(X_train[inner_train_idx], y_train[inner_train_idx])
            all_inner_scores[p_i, j] = clf_inner.predict_proba(X_train[inner_val_idx])[0, 1]

    # -- Pour chaque combo params : tune threshold + calcule bal_acc --
    best_combo_idx, best_combo_bal_acc = 0, 0.0
    best_threshold_per_combo = np.full(N_ITER, 0.5)

    for p_i in range(N_ITER):
        scores_pi = all_inner_scores[p_i]
        best_t, best_ba = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.01):
            preds = (scores_pi >= t).astype(int)
            ba = balanced_accuracy_score(inner_true_all, preds)
            if ba > best_ba:
                best_ba, best_t = ba, t
        best_threshold_per_combo[p_i] = best_t
        if best_ba > best_combo_bal_acc:
            best_combo_bal_acc = best_ba
            best_combo_idx     = p_i

    best_params    = PARAM_LIST[best_combo_idx]
    best_threshold = best_threshold_per_combo[best_combo_idx]
    thresholds_used.append(best_threshold)
    best_params_per_fold.append(best_params)

    # -- Fit final sur tout le train set --
    clf = xgb.XGBClassifier(**best_params, random_state=42, verbosity=0)
    clf.fit(X_train, y_train)

    score = clf.predict_proba(X_test)[0, 1]
    y_scores.append(score)
    y_true.append(y_test[0])
    y_pred.append(int(score >= best_threshold))

y_true, y_pred, y_scores = map(np.array, [y_true, y_pred, y_scores])

print(f'  Recall            : {recall_score(y_true, y_pred):.4f}')
print(f'  Precision         : {precision_score(y_true, y_pred):.4f}')
print(f'  Balanced Accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}')
print(f'  AUC-ROC           : {roc_auc_score(y_true, y_scores):.4f}')

print(f"\nThreshold moyen : {np.mean(thresholds_used):.3f}")
print(f"Threshold std   : {np.std(thresholds_used):.3f}")
OPTIMAL_THRESHOLD = np.mean(thresholds_used)

KeyboardInterrupt: 

In [ ]:
# Analyse des hyperparams sélectionnés par fold
params_df = pd.DataFrame(best_params_per_fold)
print("Hyperparams — stats sur les 255 folds :")
print(params_df.describe().round(3))

In [ ]:
final_clf = xgb.XGBClassifier(
    max_depth        = XXXXXXXXXX,
    n_estimators     = XXXXXXXXXX,
    learning_rate    = XXXXXXXXXX,
    reg_lambda       = XXXXXXXXXX,
    random_state=42, verbosity=0
)

final_clf.fit(X, y)

#final_clf.save_model('dyslexia_xgb_final.json')